Importamos as bibliotecas usadas para ler os arquivos, organizar as tabelas e realizar os cálculos numéricos.

In [ ]:
# Bibliotecas usadas na leitura dos dados e nos calculos.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display


Lemos as datas e os critérios do experimento no arquivo de parâmetros. A formação usa 2017 a 2019; a avaliação de 2020 a 2022 será realizada por outro integrante.

In [ ]:
# Le as datas e os criterios definidos para o experimento.
arquivo_parametros = Path('parametros_experimento.json')
with arquivo_parametros.open(encoding='utf-8') as arquivo:
    parametros = json.load(arquivo)

inicio_formacao = parametros['inicio_formacao']
fim_formacao = parametros['fim_formacao']
inicio_avaliacao = parametros['inicio_avaliacao']
fim_avaliacao = parametros['fim_avaliacao']

# Mostra os periodos sem misturar mensagens sobre arquivos e downloads.
print('Formacao:', inicio_formacao, 'a', fim_formacao)
print('Avaliacao:', inicio_avaliacao, 'a', fim_avaliacao)


Executamos o notebook de preparação financeira para disponibilizar os preços, a lista de ações e as funções auxiliares. Os preços locais são reutilizados quando já existem; sem essa cópia, o notebook consulta a fonte.

In [ ]:
# Carrega os precos e as funcoes do notebook de preparacao financeira.
%run ./Tabelas_Acoes.ipynb


Preparamos os retornos logarítmicos e a tabela de elegibilidade usando apenas informações disponíveis até o fim da formação. Os critérios consideram cobertura, variância, presença de histórico e preços inválidos; as datas da base servem como referência provisória de sessões, sem comprovar a data oficial do IPO.

In [ ]:
def preparar_formacao(precos, inicio, fim, cobertura_minima=0.9,
                      tickers_solicitados=None, sessoes=None):
    # Limita os precos ao passado e preserva a tabela original.
    inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
    precos = precos.sort_index().loc[:fim].copy()
    tickers = set(precos.columns)
    if tickers_solicitados is not None:
        tickers.update(tickers_solicitados)
    precos = precos.reindex(columns=sorted(tickers))

    # Usa as datas da base, ou o calendario recebido, como referencia.
    referencia = 'datas da base (provisorio)' if sessoes is None else 'calendario fornecido'
    if sessoes is None:
        sessoes = precos.index
    sessoes = pd.DatetimeIndex(sessoes).unique().sort_values()
    sessoes = sessoes[(sessoes >= inicio) & (sessoes <= fim)]
    precos = precos.reindex(precos.index.union(sessoes).sort_values())

    # Calcula retornos somente para fechamentos positivos e finitos.
    invalidos = precos.notna() & (~np.isfinite(precos) | (precos <= 0))
    precos_validos = precos.where(np.isfinite(precos) & (precos > 0))
    precos_janela = precos_validos.reindex(sessoes)
    retornos = np.log(precos_validos).diff().reindex(sessoes)

    # Resume a disponibilidade de dados de cada acao.
    quantidade_retornos = retornos.count()
    primeiras_datas = precos_janela.apply(lambda serie: serie.first_valid_index())
    ultimas_datas = precos_janela.apply(lambda serie: serie.last_valid_index())
    tabela = pd.DataFrame({
        'Primeiro_preco_na_formacao': primeiras_datas,
        'Ultimo_preco_na_formacao': ultimas_datas,
        'Historico_no_inicio': precos_validos.loc[:sessoes[0]].notna().any(),
        'Preco_na_ultima_sessao': precos_janela.iloc[-1].notna(),
        'Retornos_validos': quantidade_retornos,
        'Sessoes_referencia': len(sessoes),
        'Cobertura': quantidade_retornos / len(sessoes),
        'Variancia': retornos.var(),
        'Precos_invalidos': invalidos.reindex(sessoes).sum()
    })
    tabela.index.name = 'Ticker'
    tabela['Situacao'] = 'Elegivel'
    tabela['Motivo'] = 'Atende aos criterios observaveis na base.'

    # Aplica as regras da pesquisa e registra o motivo de cada decisao.
    for ticker, linha in tabela.iterrows():
        exclusoes, pendencias = [], []
        if linha['Retornos_validos'] < 2:
            exclusoes.append('Menos de dois retornos validos')
        elif not np.isfinite(linha['Variancia']) or linha['Variancia'] <= 0:
            exclusoes.append('Variancia nula ou indefinida')
        if linha['Cobertura'] < cobertura_minima:
            exclusoes.append('Cobertura abaixo do minimo')
        if not linha['Historico_no_inicio']:
            pendencias.append('Sem evidencia de negociacao no inicio')
        if not linha['Preco_na_ultima_sessao']:
            pendencias.append('Sem preco na ultima sessao')
        if linha['Precos_invalidos'] > 0:
            pendencias.append('Precos nao positivos ou infinitos')
        if exclusoes or pendencias:
            tabela.loc[ticker, 'Situacao'] = 'Excluida' if exclusoes else 'Pendente'
            tabela.loc[ticker, 'Motivo'] = '; '.join(exclusoes + pendencias)

    tabela.attrs['referencia_calendario'] = referencia
    return tabela, retornos


Aplicamos os critérios à base carregada e separamos os retornos das ações elegíveis. A tabela mostra quais ações foram incluídas, excluídas ou ficaram pendentes, com o motivo de cada decisão.

In [ ]:
# Prepara os retornos e a tabela de elegibilidade com os dados carregados.
tabela_elegibilidade, retornos_formacao = preparar_formacao(
    df['Close'],
    inicio_formacao,
    fim_formacao,
    cobertura_minima=parametros['cobertura_minima_formacao'],
    tickers_solicitados=acoes
)

# Mantem somente as acoes que atendem aos criterios da formacao.
acoes_elegiveis = tabela_elegibilidade.index[
    tabela_elegibilidade['Situacao'] == 'Elegivel'
].tolist()
retornos_formacao_elegiveis = retornos_formacao[acoes_elegiveis].copy()

print(tabela_elegibilidade['Situacao'].value_counts())
display(tabela_elegibilidade)


Calculamos a correlação de Pearson entre os retornos das ações elegíveis, exigindo o mínimo configurado de observações conjuntas por par. O padrão é 80% das datas da formação.

In [ ]:
# Exige observacoes conjuntas em pelo menos 80% das datas da formacao.
proporcao_minima = parametros['proporcao_minima_observacoes_conjuntas']
min_observacoes_conjuntas = int(np.ceil(
    len(retornos_formacao_elegiveis) * proporcao_minima
))

# Calcula Pearson entre os retornos das acoes elegiveis.
matriz_correlacoes_formacao = retornos_formacao_elegiveis.corr(
    method='pearson',
    min_periods=min_observacoes_conjuntas
)
matriz_correlacoes_formacao


Transformamos as correlações em distâncias de Mantegna. Cada ação representa um vértice, e cada distância conhecida será o peso de uma aresta.

In [ ]:
def matriz_mantegna(matriz_pearson):
    # Correlacoes maiores produzem distancias menores.
    return np.sqrt(2 * (1 - matriz_pearson))

matriz_mantegna_formacao = matriz_mantegna(matriz_correlacoes_formacao)
matriz_mantegna_formacao


Mantemos esta análise de IPCA como exploração separada. Ela correlaciona níveis de preços corrigidos e não participa da MST principal; pode ser ignorada ao executar o experimento da formação.

In [ ]:
# Mantem a analise exploratoria de IPCA separada da MST principal.
dataframe_IPCA = inflacao_mensal('01/01/2008', '31/12/2025', variacao_logaritma)
matriz_correlacao_IPCA = dataframe_IPCA.corr(method='pearson')
matriz_mantegna_IPCA = matriz_mantegna(matriz_correlacao_IPCA)
matriz_mantegna_IPCA


Criamos uma lista de arestas no formato peso, vértice inicial e vértice final. Cada par aparece uma vez, distâncias ausentes são ignoradas e a lista é ordenada por peso crescente.

In [ ]:
def criar_arestas(df):
    arestas = []
    n = len(df.columns)

    # Percorre apenas um triangulo da matriz, sem repetir pares.
    for j in range(1, n):
        for i in range(j):
            peso = df.iloc[i, j]
            # Uma distancia desconhecida nao representa uma aresta de peso zero.
            if np.isfinite(peso):
                arestas.append((peso, i, j))

    # A tupla desempata pesos iguais pelos indices dos vertices.
    return sorted(arestas)

arestas = criar_arestas(matriz_mantegna_formacao)
# Mostra uma amostra para nao imprimir milhares de arestas.
arestas[:10]


Construímos a MST com Kruskal e Union-Find. As arestas são consideradas em ordem crescente de peso e aceitas quando ligam componentes diferentes, evitando ciclos; a entrada deve permitir conectar todos os vértices.

In [ ]:
def find(lista, vertice):
    # Segue os pais ate encontrar o representante da componente.
    while lista[vertice] != vertice:
        vertice = lista[vertice]
    return vertice


def union(lista, vertice1, vertice2):
    # Liga os representantes de duas componentes diferentes.
    pai1 = find(lista, vertice1)
    pai2 = find(lista, vertice2)
    if pai1 != pai2:
        lista[pai2] = pai1


def kruskal(df):
    arestas = criar_arestas(df)
    n = len(df.columns)
    pais = list(range(n))
    mst = []

    for peso, vertice1, vertice2 in arestas:
        # Aceita a aresta quando ela une componentes, evitando ciclos.
        if find(pais, vertice1) != find(pais, vertice2):
            union(pais, vertice1, vertice2)
            mst.append((peso, vertice1, vertice2))
        # Uma arvore com n vertices possui n - 1 arestas.
        if len(mst) == n - 1:
            break

    return mst


Prim cresce a árvore a partir do vértice zero, escolhendo a menor aresta que alcança um vértice ainda não visitado. A função permanece disponível, mas sua chamada está comentada.

In [ ]:
def prim(df):
    n = len(df.columns)
    visitados = {0}
    mst = []

    while len(visitados) < n:
        menor_peso = (np.inf, None, None)
        for u in visitados:
            for v in range(n):
                if v not in visitados:
                    peso = df.iloc[u, v]
                    # Busca a menor ligacao entre visitados e nao visitados.
                    if np.isfinite(peso) and peso < menor_peso[0]:
                        menor_peso = (peso, u, v)

        # Encerra quando nao existe outra ligacao disponivel.
        if menor_peso[2] is None:
            break
        visitados.add(menor_peso[2])
        mst.append(menor_peso)

    return mst


Executamos Kruskal sobre a matriz da formação e mostramos a quantidade de arestas e o peso total. Prim não é executado nesta célula.

In [ ]:
# A chamada de Prim permanece desativada.
# mst_prim = prim(matriz_mantegna_formacao)

# Constroi a MST e soma os pesos das arestas escolhidas.
mst_kruskal = kruskal(matriz_mantegna_formacao)
peso_kruskal = sum(peso for peso, u, v in mst_kruskal)

print('Arestas Kruskal:', len(mst_kruskal))
print('Peso total Kruskal:', peso_kruskal)


Representamos cada aresta nos dois sentidos e calculamos grau, distância, excentricidade, diâmetro e centro. As distâncias somam os pesos ao longo do caminho único de uma árvore conexa; as funções pressupõem essa entrada.

In [ ]:
def criar_dicionario_adjacencia(mst):
    adjacencia = {}
    for peso, v1, v2 in mst:
        # Registra os dois sentidos de cada aresta nao dirigida.
        adjacencia.setdefault(v1, []).append((peso, v2))
        adjacencia.setdefault(v2, []).append((peso, v1))
    return adjacencia


def calcular_graus(adjacencia):
    # O grau de um vertice e a quantidade de vizinhos.
    graus = {}
    for vertice in adjacencia:
        graus[vertice] = len(adjacencia[vertice])
    return graus


def calcular_excentricidade(adjacencia):
    excentricidades = {}
    for origem in adjacencia:
        distancias = {origem: 0}
        pilha = [origem]

        # DFS: soma os pesos ao longo do caminho unico da arvore.
        while pilha:
            vertice = pilha.pop()
            for peso, vizinho in adjacencia[vertice]:
                if vizinho not in distancias:
                    distancias[vizinho] = distancias[vertice] + peso
                    pilha.append(vizinho)

        excentricidades[origem] = max(distancias.values())
    return excentricidades


def distancia_entre_vertices(adjacencia, origem, destino):
    distancias = {origem: 0}
    pilha = [origem]
    while pilha:
        vertice = pilha.pop()
        if vertice == destino:
            return distancias[vertice]
        for peso, vizinho in adjacencia[vertice]:
            if vizinho not in distancias:
                distancias[vizinho] = distancias[vertice] + peso
                pilha.append(vizinho)


def calcular_diametro(adjacencia):
    # O diametro e a maior excentricidade da arvore.
    excentricidades = calcular_excentricidade(adjacencia)
    return max(excentricidades.values())


def calcular_centro(adjacencia):
    excentricidades = calcular_excentricidade(adjacencia)
    menor_excentricidade = min(excentricidades.values())
    centro = []
    for vertice, excentricidade in excentricidades.items():
        # A tolerancia considera pequenos arredondamentos nas somas.
        if abs(excentricidade - menor_excentricidade) <= 1e-10:
            centro.append(vertice)
    return centro


Reunimos as métricas dos vértices e associamos seus índices aos tickers da mesma matriz usada em Kruskal. A tabela mostra grau, excentricidade e identificação de folhas e centro.

In [ ]:
# Calcula as metricas usando a arvore produzida por Kruskal.
adjacencia = criar_dicionario_adjacencia(mst_kruskal)
graus = calcular_graus(adjacencia)
excentricidades = calcular_excentricidade(adjacencia)
diametro = calcular_diametro(adjacencia)
centro = calcular_centro(adjacencia)
vertices = sorted(adjacencia)

# Recupera os tickers na mesma ordem da matriz usada no grafo.
metricas = pd.DataFrame({
    'Vertice': vertices,
    'Ticker': [matriz_mantegna_formacao.columns[v] for v in vertices],
    'Grau': [graus[v] for v in vertices],
    'Excentricidade': [excentricidades[v] for v in vertices],
    'Folha': [graus[v] == 1 for v in vertices],
    'Centro': [v in centro for v in vertices]
})

display(metricas.sort_values(['Excentricidade', 'Ticker']))
print('Diametro:', diametro)
print('Centro:', [matriz_mantegna_formacao.columns[v] for v in centro])


Selecionamos as oito ações com menores excentricidades para a carteira central. Empates são resolvidos pelo ticker em ordem alfabética, preservando as demais métricas.

In [ ]:
# Ordena da menor para a maior excentricidade, com desempate por ticker.
carteira_central = metricas.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[True, True]
).head(8).copy()

display(carteira_central)


Selecionamos as oito ações com maiores excentricidades para a carteira periférica. Empates são resolvidos pelo ticker em ordem alfabética; as duas seleções são independentes e não removem eventual sobreposição.

In [ ]:
# Ordena da maior para a menor excentricidade, com desempate por ticker.
carteira_periferica = metricas.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[False, True]
).head(8).copy()

display(carteira_periferica)


A carteira de controle usa os ativos presentes na MST que pertenciam à carteira recomendada pelo BTG Pactual em abril de 2019, descrita no relatório como mais defensiva e diversificada. A escolha vem da referência externa, sem grau, excentricidade, distância ou desempenho de 2020 a 2022; o filtro da MST considera somente os tickers. Os segmentos seguem o relatório, não uma classificação setorial oficial da B3. A interseção adapta a carteira original e não constitui comprovação de proteção em crises. Fonte: https://content.btgpactual.com/wp-content/uploads/2019/04/10_sim_abr_2019-vfinal.pdf

In [ ]:
# Referencia externa publicada antes da avaliacao da pandemia.
# Segmentos informados no relatorio do BTG Pactual de abril de 2019.
fonte_controle = 'https://content.btgpactual.com/wp-content/uploads/2019/04/10_sim_abr_2019-vfinal.pdf'
referencia_controle = pd.DataFrame([
    ('PETR4.SA', 'Oleo e Gas'),
    ('BBDC4.SA', 'Bancos'),
    ('CSAN3.SA', 'Acucar e Etanol'),
    ('IRBR3.SA', 'Financeiro'),
    ('LREN3.SA', 'Varejo'),
    ('RAIL3.SA', 'Infraestrutura'),
    ('RENT3.SA', 'Aluguel de carros'),
    ('SUZB3.SA', 'Papel e Celulose'),
    ('ABEV3.SA', 'Alimentos e Bebidas'),
    ('OIBR3.SA', 'Telecom')
], columns=['Ticker', 'Segmento'])

# Consulta apenas a presenca dos ativos na arvore, sem usar suas metricas.
tickers_mst = {
    matriz_mantegna_formacao.columns[vertice]
    for _, u, v in mst_kruskal
    for vertice in (u, v)
}
presentes = referencia_controle['Ticker'].isin(tickers_mst)
carteira_controle = referencia_controle.loc[presentes].copy()
carteira_controle = carteira_controle.sort_values('Ticker').reset_index(drop=True)

# Mostra a adaptacao da referencia, sem substituir os ativos ausentes.
print('Referencia: BTG Pactual, abril de 2019')
print('Fonte:', fonte_controle)
print('Papeis da referencia fora da MST:', referencia_controle.loc[~presentes, 'Ticker'].tolist())
display(carteira_controle)
